In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Етапи транспілятора

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.4.0
```
</details>
На цій сторінці описані етапи вбудованого конвеєра транспіляції в Qiskit SDK. Існує шість етапів:

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

Функція [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) створює наперед визначений [поетапний менеджер проходів](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager), що складається з цих етапів. Конкретні проходи, що складають кожен етап, залежать від аргументів, переданих у `generate_preset_pass_manager`. `optimization_level` — це позиційний аргумент, який необхідно задати; це ціле число, що може приймати значення 0, 1, 2 або 3. Вищі значення означають більш ресурсоємну, але якіснішу оптимізацію (дивись [Стандартні налаштування та параметри конфігурації транспіляції](defaults-and-configuration-options)).

Рекомендований спосіб транспіляції схеми — створити наперед визначений поетапний менеджер проходів і запустити його на схемі, як описано в розділі [Транспіляція за допомогою менеджерів проходів](transpile-with-pass-managers). Однак простішою, але менш гнучкою альтернативою є функція [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile). Ця функція приймає схему безпосередньо як аргумент. Як і у випадку з `generate_preset_pass_manager`, конкретні проходи транспілятора залежать від аргументів, таких як `optimization_level`, переданих у `transpile`. Фактично, внутрішньо функція `transpile` викликає `generate_preset_pass_manager` для створення наперед визначеного поетапного менеджера проходів і запускає його на схемі.
## Етап init
Цей перший етап за замовчуванням робить дуже мало і насамперед корисний, якщо ти хочеш включити власні початкові оптимізації. Оскільки більшість алгоритмів розмітки та маршрутизації розроблені для роботи лише з однокубітними та двокубітними вентилями, цей етап також використовується для перетворення будь-яких вентилів, що діють на більше двох кубітів, у вентилі, що діють лише на одному або двох кубітах.

Більше інформації про реалізацію власних початкових оптимізацій для цього етапу дивись у розділі про плагіни та налаштування менеджерів проходів.
## Етап layout
Наступний етап пов'язаний з розміткою або зв'язністю бекенду, на який буде надіслано схему. Загалом квантові схеми — це абстрактні сутності, кубіти яких є "віртуальними" або "логічними" представленнями реальних кубітів, що використовуються в обчисленнях. Для виконання послідовності вентилів необхідне взаємно однозначне відображення "віртуальних" кубітів на "фізичні" кубіти реального квантового пристрою. Це відображення зберігається у вигляді об'єкта `Layout` і є частиною обмежень, визначених в [архітектурі набору інструкцій (ISA)](./transpile#instruction-set-architecture) бекенду.

![Це зображення ілюструє відображення кубітів від дротової схеми до діаграми, що показує, як кубіти підключені на QPU.](../docs/images/guides/transpiler-stages/layout-mapping.avif "Відображення кубітів")

Вибір відображення є надзвичайно важливим для мінімізації кількості операцій SWAP, необхідних для відображення вхідної схеми на топологію пристрою, і для забезпечення використання найкраще скаліброваних кубітів. Зважаючи на важливість цього етапу, наперед визначені менеджери проходів пробують кілька різних методів для пошуку найкращої розмітки. Як правило, це передбачає два кроки: спочатку спробувати знайти "ідеальну" розмітку (розмітку, що не вимагає жодних операцій SWAP), а потім евристичний прохід, який намагається знайти найкращу розмітку, якщо ідеальну знайти не вдається. Для першого кроку зазвичай використовуються два проходи `Passes`:

- `TrivialLayout`: Наївно відображає кожен віртуальний кубіт на фізичний кубіт з тим самим номером на пристрої (тобто [`0`,`1`,`1`,`3`] -> [`0`,`1`,`1`,`3`]). Це поведінка, що збереглась з попередніх версій і використовується лише в `optimization_level=1` для спроби знайти ідеальну розмітку. Якщо це не вдається, наступним пробується `VF2Layout`.
- `VF2Layout`: Це `AnalysisPass`, що вибирає ідеальну розмітку, трактуючи цей етап як задачу ізоморфізму підграфів, розв'язану алгоритмом VF2++. Якщо знайдено більше однієї розмітки, запускається евристична оцінка для вибору відображення з найнижчою середньою похибкою.

Потім для евристичного етапу за замовчуванням використовуються два проходи:

- `DenseLayout`: Знаходить підграф пристрою з найбільшою зв'язністю, що має таку саму кількість кубітів, як і схема (використовується для рівня оптимізації 1, якщо в схемі є операції управляючого потоку, наприклад `IfElseOp`).
- `SabreLayout`: Цей прохід вибирає розмітку, починаючи з початкової випадкової розмітки та багаторазово запускаючи алгоритм `SabreSwap`. Цей прохід використовується лише на рівнях оптимізації 1, 2 та 3, якщо ідеальна розмітка не знайдена за допомогою проходу `VF2Layout`. Детальніше про цей алгоритм дивись у статті [arXiv:1809.02573](https://arxiv.org/abs/1809.02573).
## Етап routing
Щоб реалізувати двокубітний вентиль між кубітами, які безпосередньо не з'єднані на квантовому пристрої, до схеми необхідно вставити один або кілька вентилів SWAP для переміщення станів кубітів до тих пір, поки вони не стануть суміжними на карті вентилів пристрою. Кожен вентиль SWAP є дорогою та шумною операцією для виконання. Тому пошук мінімальної кількості вентилів SWAP, необхідних для відображення схеми на даний пристрій, є важливим кроком у процесі транспіляції. З міркувань ефективності цей етап зазвичай за замовчуванням обчислюється разом з етапом Layout, але вони логічно відрізняються один від одного. Етап *Layout* вибирає апаратні кубіти для використання, тоді як етап *Routing* вставляє відповідну кількість вентилів SWAP для виконання схем з вибраною розміткою.

Однак пошук оптимального відображення SWAP є складним завданням. Фактично це NP-важка задача, і тому її обчислення є надто дорогим для всіх, крім найменших квантових пристроїв та вхідних схем. Щоб обійти це, Qiskit використовує стохастичний евристичний алгоритм `SabreSwap` для обчислення хорошого, але не обов'язково оптимального, відображення SWAP. Використання стохастичного методу означає, що генеровані схеми не гарантовано будуть однаковими при повторних запусках. Справді, повторне виконання однієї й тієї ж схеми дає розподіл глибин схем і кількості вентилів на виході. Саме тому багато користувачів вирішують запускати функцію маршрутизації (або весь `StagedPassManager`) багато разів і вибирати схеми з найменшою глибиною з розподілу результатів.

Наприклад, розглянемо 15-кубітну схему GHZ, виконану 100 разів із "поганою" (незв'язаною) `initial_layout`.

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.providers.fake_provider import GenericBackendV2

backend = GenericBackendV2(15)


ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/62479697-cef1-4a89-ac2a-20051dd294f4-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ab89c4ea-06c4-4320-b493-feb691b3570d-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/f6a3a92a-8656-4518-ba2c-c3b0b038f507-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](/docs/guides/transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/d4d1f65a-3336-4d70-9189-65ba010f2366-1.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg)

Як видно, ця схема повинна виконати двокубітний вентиль між кубітами 0 та 14, які розташовані дуже далеко один від одного на графі зв'язності. Тому виконання цієї схеми вимагає вставки вентилів SWAP для виконання всіх двокубітних вентилів за допомогою проходу `SabreSwap`.

Зауваж також, що алгоритм `SabreSwap` відрізняється від ширшого методу `SabreLayout` на попередньому етапі. За замовчуванням `SabreLayout` виконує як розмітку, так і маршрутизацію, і повертає трансформовану схему. Це робиться з кількох конкретних технічних причин, зазначених на [сторінці API-довідника](../api/qiskit/qiskit.transpiler.passes.SabreLayout) проходу.
## Етап translation
При написанні квантової схеми ти можеш використовувати будь-який квантовий вентиль (унітарну операцію), а також колекцію невентильних операцій, таких як інструкції вимірювання або скидання кубітів. Однак більшість квантових пристроїв нативно підтримують лише невелику кількість квантових вентильних і невентильних операцій. Ці нативні вентилі є частиною визначення [ISA](./transpile#instruction-set-architecture) цілі, і цей етап наперед визначених `PassManagers` перекладає (або *розгортає*) вентилі, задані в схемі, до нативних базових вентилів вказаного бекенду. Це важливий крок, оскільки він дозволяє схемі виконуватися бекендом, але зазвичай призводить до збільшення глибини та кількості вентилів.

Два особливих випадки заслуговують на особливу увагу і допомагають проілюструвати, що робить цей етап.

1. Якщо вентиль SWAP не є нативним вентилем цільового бекенду, для його реалізації потрібні три вентилі CNOT:

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4552b367-75e9-4faa-a59d-8317e25ac145-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2700405a-f559-45d3-99a9-5b4447621743-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg)

Будучи добутком трьох вентилів CNOT, SWAP є дорогою операцією для виконання на шумних квантових пристроях. Однак такі операції зазвичай необхідні для вбудовування схеми в обмежену зв'язність вентилів багатьох пристроїв. Тому мінімізація кількості вентилів SWAP у схемі є основною метою процесу транспіляції.

2. Вентиль Тоффолі, або вентиль controlled-controlled-not (`ccx`), є трикубітним вентилем. Враховуючи, що наш базовий набір вентилів включає лише однокубітні та двокубітні вентилі, ця операція повинна бути розкладена. Однак це досить затратно:

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = GenericBackendV2(5)

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ae2d9390-5c26-46f3-9418-a684ba8a406a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg)

Для кожного вентиля Тоффолі в квантовій схемі апаратне забезпечення може виконувати до шести вентилів CNOT і кілька однокубітних вентилів. Цей приклад демонструє, що будь-який алгоритм, що використовує кілька вентилів Тоффолі, матиме велику глибину схеми і тому суттєво постраждає від шуму.
## Етап optimization
Цей етап зосереджений на розкладенні квантових схем до базового набору вентилів цільового пристрою і повинен протистояти збільшенню глибини від етапів розмітки та маршрутизації. На щастя, існує багато підпрограм для оптимізації схем шляхом комбінування або видалення вентилів. У деяких випадках ці методи настільки ефективні, що вихідні схеми мають меншу глибину, ніж вхідні, навіть після розмітки та маршрутизації до топології апаратного забезпечення. В інших випадках мало що можна зробити, і обчислення може бути важким для виконання на шумних пристроях. Саме на цьому етапі різні рівні оптимізації починають відрізнятися.

- Для `optimization_level=1` цей етап підготовлює [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) та [`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), які комбінують ланцюжки однокубітних вентилів і скасовують будь-які пари вентилів CNOT, що стоять поруч.
- Для `optimization_level=2` цей етап використовує прохід [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) замість `CXCancellation`, який видаляє надлишкові вентилі, використовуючи відношення комутативності.
- Для `optimization_level=3` цей етап підготовлює наступні проходи:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

Крім того, цей етап також виконує кілька фінальних перевірок, щоб переконатися, що всі інструкції в схемі складаються з базових вентилів, доступних на цільовому бекенді.

Наведений нижче приклад зі станом GHZ демонструє ефекти різних рівнів оптимізації на глибину схеми та кількість вентилів.

> **Note:** Результат транспіляції варіюється через стохастичний SWAP-мапер. Тому наведені нижче числа, ймовірно, зміняться кожного разу при запуску коду.

![15-кубітний стан GHZ](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-кубітний стан GHZ до транспіляції")

Наступний код конструює 15-кубітний стан GHZ і порівнює `optimization_levels` транспіляції з точки зору отриманої глибини схеми, кількості вентилів і кількості багатокубітних вентилів.